In [1]:
from pathlib import Path
import os

### Preparation

The Boltz calculation requires two parts, a **multi-sequence alignment (MSA)** step and an inference step. Boltz relies on external programs and servers to run the MSA. Here we cannot use an MSA server, but have to provide a precomputed MSA file in a3m or csv format. For each protein entity in the simulation the MSA needs to be computed individually beforehand!

Next we need to define where AlphaFold finds our input data and where the output files are written to. You can see these files in the file browser on the left. If you change these names, remember to change them in the analysis notebook as well.

In [2]:
BOLTZ_WORKING_DIR = Path.home() / "boltz_test"

Now we need to prepare the input file. Remember to link the MSA files to the respective sequence entries! 

For this example you can download the example.a3m file from our [GitHub](https://github.com/ssciwr/BioStructureHub).

In [3]:
MSA_PATH = BOLTZ_WORKING_DIR / "example.a3m"
INPUT_FILE = BOLTZ_WORKING_DIR / "input_file.yaml"

BOLTZ_RUN_FILE = "run.sh"
BOLTZ_RUN_PATH = BOLTZ_WORKING_DIR / BOLTZ_RUN_FILE  # will be created in this notebook

In [4]:
test_file = f"""
version: 1
sequences:
  - protein:
      id: [A] 
      sequence: GMRESYANENQFGFKTINSDIHKIVIVGGYGKLGGLFARYLRASGYPISILDREDWAVAESILANADVVIVSVPINLTLETIERLKPYLTENMLLADLTSVKREPLAKMLEVHTGAVLGLHPMFGADIASMAKQVVVRCDGRFPERYEWLLEQIQIWGAKIYQTNATEHDHNMTYIQALRHFSTFANGLHLSKQPINLANLLALSSPIYRLELAMIGRLFAQDAELYADIIMDKSENLAVIETLKQTYDEALTFFENNDRQGFIDAFHKVRDWFGDYSEQFLKESRQLLQQANDLKQG
      msa: {MSA_PATH}
"""

with open(INPUT_FILE, "w") as text_file:
    text_file.write(test_file)

In [ ]:
run_file = f"""
#!/bin/bash

module load devel/miniforge/24.9.2
module load devel/cuda/12.8
conda activate /mnt/sds-hd/sd25g005/boltz


boltz predict  {str(INPUT_FILE)}  \\
    --write_full_pae \\
    --out_dir {BOLTZ_WORKING_DIR}
"""

with open(BOLTZ_RUN_PATH, "w") as file:
    file.write(run_file)
    print(f"File written to {BOLTZ_RUN_PATH}.")

### Run the Boltz prediction

Execute the next cell to start the run! Good luck!

In [6]:
os.system(f'echo "Running file {BOLTZ_RUN_PATH}"')
os.system(f"bash {BOLTZ_RUN_PATH}")

Running file /home/hd/hd_hd/hd_aq354/boltz_test/run.sh
Checking input data.
Processing 1 inputs with 1 threads.


100%|██████████| 1/1 [00:00<00:00,  2.13it/s]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/hd/hd_hd/hd_aq354/.local/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


Running structure prediction for 1 input.


/home/hd/hd_hd/hd_aq354/.local/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
You are using a CUDA device ('NVIDIA A40') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.
/mnt/sds-hd/sd25g005/boltz/lib/python3.12/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 2 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive wo

Predicting DataLoader 0: 100%|██████████| 1/1 [00:13<00:00,  0.08it/s]


0

Done! 

Number of failed examples should be 0. Also check if there is a warning "MSA does not match input sequence" that indicates that something went wrong with the precomputed msa.